In [ ]:
import pandas as pd
%load_ext autoreload
%autoreload 2

from pathlib import Path
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import napari
import colorcet as cc

import dnt

spots_directory = Path(r"C:\Tracking\BlastodermAnalysis\data\spots")
embryo_overview = pd.read_excel(spots_directory / "overview.xlsx", sheet_name="Sheet1")
save_path = Path(r"C:\Tracking\BlastodermAnalysis\figures\all_napari_movies")


embryo_overview = embryo_overview[embryo_overview["good"]]
included = embryo_overview["Embryo"].astype(str).tolist()
condition_map = {
    str(embryo): condition for embryo, condition in zip(embryo_overview["Embryo"], embryo_overview["condition"])
}
print(condition_map)

dnt.set_plot_style()
spots_dfs, stems = dnt.load_spots_data(spots_directory, included)

print(stems)

df = spots_dfs[0]
cycles = [10, 11, 12, 13, 14]

print(df.columns)

greens = ["#143601","#1a4301","#245501","#538d22","#73a942","#aad576"][::-1]
blues = ["#012a4a","#01497c","#2a6f97","#468faf","#89c2d9"][::-1]
reds = ["#641220","#85182a","#a71e34","#bd1f36", "#da1e37"][::-1]
oranges = ["#fbba72","#ca5310","#bb4d00","#8f250c","#691e06"]
condition_pal_map = {
    "wt": blues,
    "bcd": reds,
    "trk": greens,
}
condition_main_colors = {
    c: cmap[2] for c, cmap in condition_pal_map.items()
}
n = 3
condition_pal = {
    "wt": blues[n],
    "bcd": reds[n],
    "trk": greens[n],
}

In [ ]:
import cv2
import ffmpegcv
from tqdm import tqdm
import colorcet as cc

def napari_to_mp4(napari_viewer, save_file, fps=30):

    frames = []
    for i in tqdm(range(napari_viewer.dims.nsteps[0])):
        napari_viewer.dims.set_current_step(0, i)
        frame = napari_viewer.screenshot()
        frames.append(frame)

    height, width, _ = frames[0].shape
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    video_writer = cv2.VideoWriter(str(save_file), fourcc, fps, (width, height))

    if not video_writer.isOpened():
        raise RuntimeError("VideoWriter failed to open — check codec and output path")

    for frame in frames:
        frame = cv2.cvtColor(frame, cv2.COLOR_RGBA2BGR)
        video_writer.write(frame)

    video_writer.release()





In [ ]:
import subprocess
import os

def napari_to_mp4(napari_viewer, save_file, fps=30):
    temp_path = str(save_file).replace('.mp4', '_temp.mp4')

    frames = []
    for i in tqdm(range(napari_viewer.dims.nsteps[0])):
        napari_viewer.dims.set_current_step(0, i)
        frame = napari_viewer.screenshot()
        frames.append(frame)

    height, width, _ = frames[0].shape
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    video_writer = cv2.VideoWriter(temp_path, fourcc, fps, (width, height))

    if not video_writer.isOpened():
        raise RuntimeError("VideoWriter failed to open")

    for frame in tqdm(frames):
        frame_bgr = cv2.cvtColor(frame, cv2.COLOR_RGBA2BGR)
        video_writer.write(frame_bgr)

    video_writer.release()

    # Re-encode to H.264 — typically 5-10x smaller than mp4v
    subprocess.run([
        'ffmpeg', '-y',           # -y overwrites output if it exists
        '-i', temp_path,
        '-vcodec', 'libx264',
        '-crf', '18',             # 0=lossless, 51=worst; 18-28 is a good range
        '-preset', 'slow',        # slower preset = better compression
        str(save_file)
    ], check=True)

    os.remove(temp_path)
    print(f"Saved to {save_file}")

In [ ]:
def get_camera_normal(viewer):
    """Return unit normal of the current camera-facing plane (world space)."""
    # napari camera angles: azimuth rotates around z, elevation tilts up
    az = np.radians(viewer.camera.angles[2])   # azimuth
    el = np.radians(viewer.camera.angles[0])   # elevation
    # Convert spherical → Cartesian (napari uses z-up convention)
    nx = np.cos(el) * np.sin(az)
    ny = np.sin(el)
    nz = np.cos(el) * np.cos(az)
    return np.array([nx, ny, nz])

def signed_distance_from_plane(points, plane_point, normal):
    """Signed distance of each point from a plane defined by a point + normal."""
    n = normal / np.linalg.norm(normal)
    return (points[..., :3] - plane_point) @ n

def opacity_from_distance(distances, cutoff=0.0, falloff=50.0, min_alpha=0.1):
    """
    Points beyond `cutoff` fade toward `min_alpha` over `falloff` units.
    Points at or in front of the plane stay fully opaque.
    """
    beyond = np.clip(distances - cutoff, 0, None)          # only positive = behind plane
    alpha = 1.0 - (1.0 - min_alpha) * np.clip(beyond / falloff, 0, 1)
    return alpha.astype(np.float32)

def update_point_opacity(points, pts_layer, colors, napari_viewer, plane_distance_from_camera=80.0, falloff=40.0, min_alpha=0.05):
    """
    Recompute per-point alpha based on current camera orientation.

    plane_distance_from_camera: how far in front of the camera centre the
                                 cutoff plane sits (world units)
    falloff:  distance over which opacity fades after the plane
    min_alpha: minimum opacity for far-away points
    """
    cam_center = np.array(viewer.camera.center)
    normal = get_camera_normal(viewer)

    # Plane anchor = camera center shifted forward along view direction
    plane_point = cam_center + normal * plane_distance_from_camera

    dists = -signed_distance_from_plane(points, plane_point, normal)
    print(dists.mean())
    alphas = opacity_from_distance(dists, cutoff=0.0, falloff=falloff, min_alpha=min_alpha)
    print(alphas.mean())

    # Build RGBA — reuse existing face colours, just overwrite alpha
    colors[:, 3] = alphas
    border_colors = np.zeros_like(colors)
    border_colors[:, 3] = alphas
    pts_layer.face_color = colors
    pts_layer.border_color = border_colors
    pts_layer.refresh()  # trigger napari redraw

In [ ]:
viewer = napari.Viewer(ndisplay=3)
pal = sns.color_palette("Spectral", as_cmap=True)
df = spots_dfs[0]
pos = df[["frame", "z", "y", "x"]].values

colors = [cc.glasbey_cool[tid % 255] for tid in df["track_id"]]

p_layer = viewer.add_points(pos, size=df["radius"]*2.5, face_color=colors, border_color="k")
viewer.theme = "light"

viewer.camera.angles = (86, -15, 102)
viewer.camera.zoom = 3.7

update_point_opacity(pos, p_layer, p_layer.face_color, viewer, plane_distance_from_camera=-150.0, falloff=100.0, min_alpha=0.05)



In [ ]:
print(viewer.camera.angles)
print(viewer.camera.zoom)

In [ ]:
viewer.screenshot(save_path / "glasbey_cool_track_ids.png", scale=4)

In [ ]:
napari_to_mp4(viewer, save_path / "glasbey_cool_track_ids.mp4", fps=15)

In [ ]:
viewer = napari.Viewer(ndisplay=3)
pal = sns.color_palette("Spectral", as_cmap=True)
df = spots_dfs[0]
pos = df[["frame", "z", "y", "x"]].values

cycle_pal = dnt.palettes.nc
colors = [cycle_pal[c] for c in df["cycle"]]

p_layer = viewer.add_points(pos, size=df["radius"]*2.5, face_color=colors, border_color="k")
viewer.theme = "light"

viewer.camera.angles = (86, -15, 102)
viewer.camera.zoom = 3.7

update_point_opacity(pos, p_layer, p_layer.face_color, viewer, plane_distance_from_camera=-150.0, falloff=100.0, min_alpha=0.05)



In [ ]:
napari_to_mp4(viewer, save_path / "nuclear_cycles.mp4", fps=15)

In [ ]:
viewer = napari.Viewer(ndisplay=3)
pal = sns.color_palette("Spectral", as_cmap=True)
df = spots_dfs[3]
pos = df[["frame", "z", "y", "x"]].values

movement_pal = sns.diverging_palette(220, 20, as_cmap=True)
print(df["dAP"].describe())
colors = movement_pal(np.clip(df["dAP"].values*200, -1, 1)*0.5 + 0.5)
print(colors)

p_layer = viewer.add_points(pos, size=df["radius"]*2.1, face_color=colors, border_color="k")
viewer.theme = "light"

viewer.camera.angles = (86, -15, 102)
viewer.camera.zoom = 3.7

update_point_opacity(pos, p_layer, p_layer.face_color, viewer, plane_distance_from_camera=-150.0, falloff=100.0, min_alpha=0.05)

# save colorbar
fig, ax = plt.subplots(figsize=(1, 4))
norm = mpl.colors.Normalize(vmin=-0.5, vmax=0.5)
cb1 = mpl.colorbar.ColorbarBase(ax, cmap=movement_pal, norm=norm, orientation='vertical')
cb1.set_label('%AP axis per minute')
cb1.set_ticks([-0.5, -0.25, 0, 0.25, 0.5])
plt.savefig(save_path / "dAP_colorbar.png", bbox_inches='tight')


In [ ]:
napari_to_mp4(viewer, save_path / "sloshing_bad_embryo.mp4", fps=15)

In [ ]:
viewer = napari.Viewer(ndisplay=3)
n = 50

df = spots_dfs[0]
pos = df[["frame", "z", "y", "x"]].values

track_ids = df.query("frame == frame.min()")["track_id"].unique()
track_ids = np.random.choice(track_ids, n, replace=False)
t = df.query("track_id.isin(@track_ids)").groupby("track_id")["AP"].mean()
# track_ids = sorted(t.index, key=lambda tid: t[tid])
pal = sns.color_palette("Spectral", n_colors=len(track_ids))
track_id_map = {tid: pal[i] for i, tid in enumerate(track_ids)}


colors = [track_id_map.get(tid, (0.5, 0.5, 0.5)) for tid in df["track_id"]]

p_layer = viewer.add_points(pos, size=df["radius"]*2.1, face_color=colors, border_color="k")
viewer.theme = "light"

viewer.camera.angles = (86, -15, 102)
viewer.camera.zoom = 3.7

update_point_opacity(pos, p_layer, p_layer.face_color, viewer, plane_distance_from_camera=-150.0, falloff=100.0, min_alpha=0.05)
